# 📘 Class Activity: Epsilon-Closure Computation for NFAs
### Course: Theory of Automata  |  Topic 4: Subset Construction Algorithm (NFA to DFA Conversion)
**Total Marks: 10**

**Instructions for students:**
- Read the concept explanation below carefully before attempting the question.
- Write your code only inside the section marked `# TODO: write your code here`.
- Do **not** modify the provided helper code (the NFA evaluation engine) or the test-case cell.
- Run the **Main** cell at the end to see your function in action.
- Run the test-case cell — if all assertions pass without error, your solution is correct.


## 🧠 Concept Explanation

**What is an epsilon-transition?**

In a Nondeterministic Finite Automaton (NFA), a state can move to another state **without consuming any input symbol**. This kind of move is called an **epsilon-transition** (written ε-transition), and it lets the machine "jump" between states for free.

**What is the epsilon-closure of a state?**

The **epsilon-closure** of a state `q`, written `ECLOSE(q)`, is the set of **all** states reachable from `q` using **zero or more** epsilon-transitions — this always includes `q` itself (reachable via *zero* epsilon-transitions).

- e.g. if `q0 --ε--> q1` and `q1 --ε--> q2`, then `ECLOSE(q0) = {q0, q1, q2}` (the closure is **transitive**).

**Why does this matter? (connection to Subset Construction)**

When simulating an NFA on an input string, we cannot track just a single "current state" the way we do for a DFA — at every point the machine may really be in **several states at once** (nondeterminism), and some of those states may be reachable only through epsilon-transitions. So instead of a single state, we track a **set of current states**.

After consuming each input symbol, two things happen:
1. We move to all states reachable from the current set on that symbol.
2. We then take the epsilon-closure of that resulting set, since the machine could silently take any epsilon-transitions from there.

Treating "a set of NFA states" as a single unit is exactly the core idea behind the **Subset Construction Algorithm**, which converts an NFA into an equivalent DFA by making every DFA state correspond to one such subset of NFA states. The epsilon-closure function you will write today is the essential building block that makes this possible.


## 🛠️ Provided Code For Understanding

The cell below implements an **NFA evaluation engine** that simulates an NFA on an input string by tracking a *set* of current states (multi-state transition sets), exactly as described above. It is provided so you can focus on the epsilon-closure question rather than re-deriving the whole engine from scratch.

**Note:** `nfa_accepts` relies on the function `epsilon_closure`, which you will implement below. This cell will run without errors, but `nfa_accepts` will not work correctly until you complete the Question 1 cell.


In [ ]:
EPSILON = 'epsilon'  # the symbol used to label empty (epsilon) transitions


def move(states, symbol, transitions):
    """
    Return the set of states reachable from any state in `states` by
    reading exactly one input symbol `symbol` (no epsilon-transitions).
    """
    result = set()
    for s in states:
        result |= transitions.get((s, symbol), set())
    return result


def epsilon_closure_set(states, transitions):
    """
    Return the epsilon-closure of a *set* of states, i.e. the union of
    epsilon_closure(s, transitions) for every s in `states`.
    Built on top of your epsilon_closure function.
    """
    result = set()
    for s in states:
        result |= epsilon_closure(s, transitions)
    return result


def nfa_accepts(nfa, input_string):
    """
    Simulate the NFA on `input_string` using multi-state transition sets.

    nfa: dict with keys:
        'states'      -> set of all state names
        'alphabet'    -> set of input symbols (excluding epsilon)
        'transitions' -> dict mapping (state, symbol) -> set of states
                          (symbol may be EPSILON for empty transitions)
        'start'       -> start state
        'accept'      -> set of accepting states

    Returns True if the NFA accepts input_string, else False.
    """
    current_states = epsilon_closure_set({nfa['start']}, nfa['transitions'])
    for ch in input_string:
        current_states = move(current_states, ch, nfa['transitions'])
        current_states = epsilon_closure_set(current_states, nfa['transitions'])
    return bool(current_states & nfa['accept'])


---
## ❓ Question 1 (10 Marks)

Implement the function **`epsilon_closure(state, transitions)`** which takes:
- `state`: a single NFA state (string), and
- `transitions`: the NFA's transition dictionary, where keys are `(state, symbol)` tuples and `symbol` may be `EPSILON` for empty transitions (as used by the provided engine above),

and returns the **epsilon-closure of `state`** as a Python `set`: all states reachable from `state` using zero or more epsilon-transitions.

**Requirements**
1. The returned set **must** include `state` itself (zero epsilon-transitions is always allowed).
2. The closure must be **transitive** — if `q0 --ε--> q1` and `q1 --ε--> q2`, then `epsilon_closure('q0', transitions)` must include `q2`.
3. A state with no outgoing epsilon-transitions should simply return `{state}`.
4. Your function must handle **cycles** of epsilon-transitions without looping forever (e.g. `q0 --ε--> q1 --ε--> q0`).
5. Your function must work for **any** NFA transition dictionary passed in — do not hard-code state names.
6. Return type: a `set` of state-name strings (not a list).

**Example:** if `transitions` contains the epsilon-chain `q0 --ε--> q1 --ε--> q2`, then `epsilon_closure('q0', transitions)` should return `{'q0', 'q1', 'q2'}`.


In [5]:
def epsilon_closure(state, transitions):
    """
    Compute the epsilon-closure of a single NFA state.

    Parameters
    ----------
    state : str
        The NFA state to start from, e.g. "q0"
    transitions : dict
        Maps (state, symbol) -> set of states. Epsilon-transitions are
        stored under the key (state, EPSILON).

    Returns
    -------
    set of str
        All states reachable from `state` via zero or more epsilon-transitions,
        including `state` itself.
    """
    # TODO: write your code here
def epsilon_closure(state, transitions):
    closure = {state}
    stack = [state]

    while stack:
        current = stack.pop()

        next_states = transitions.get((current, EPSILON), set())

        for next_state in next_states:
            if next_state not in closure:
                closure.add(next_state)
                stack.append(next_state)

    return closure


### ✅ Test Cases — Question 1
Run this cell after implementing `epsilon_closure`. If every `assert` passes silently, the implementation is correct.


In [6]:
# --- TA / Auto-grading test cases for Question 1 ---

# Test NFA 1: a simple epsilon chain q0 -> q1 -> q2 -> q3
transitions_chain = {
    ('q0', EPSILON): {'q1'},
    ('q1', EPSILON): {'q2'},
    ('q2', EPSILON): {'q3'},
}

# Test NFA 2: a state with no outgoing epsilon-transitions
transitions_isolated = {
    ('q0', 'a'): {'q1'},  # non-epsilon transition, should be ignored
}

# Test NFA 3: a branching epsilon structure (one state, two epsilon targets)
transitions_branch = {
    ('q0', EPSILON): {'q1', 'q2'},
    ('q1', EPSILON): {'q3'},
}

# Test NFA 4: a cycle of epsilon-transitions
transitions_cycle = {
    ('q0', EPSILON): {'q1'},
    ('q1', EPSILON): {'q2'},
    ('q2', EPSILON): {'q0'},
}

test_cases_q1 = [
    (('q0', transitions_chain), {'q0', 'q1', 'q2', 'q3'}),
    (('q3', transitions_chain), {'q3'}),
    (('q0', transitions_isolated), {'q0'}),
    (('q0', transitions_branch), {'q0', 'q1', 'q2', 'q3'}),
    (('q0', transitions_cycle), {'q0', 'q1', 'q2'}),
]

passed = 0
for (state, trans), expected in test_cases_q1:
    result = epsilon_closure(state, trans)
    status = "PASS" if result == expected else "FAIL"
    if status == "PASS":
        passed += 1
    print(f"[{status}] epsilon_closure({state!r}, ...) -> {result}  (expected {expected})")

print(f"\n{passed}/{len(test_cases_q1)} test cases passed for Question 1.")
assert passed == len(test_cases_q1), "Question 1: some test cases failed."


[PASS] epsilon_closure('q0', ...) -> {'q1', 'q0', 'q2', 'q3'}  (expected {'q1', 'q0', 'q2', 'q3'})
[PASS] epsilon_closure('q3', ...) -> {'q3'}  (expected {'q3'})
[PASS] epsilon_closure('q0', ...) -> {'q0'}  (expected {'q0'})
[PASS] epsilon_closure('q0', ...) -> {'q1', 'q0', 'q2', 'q3'}  (expected {'q1', 'q0', 'q2', 'q3'})
[PASS] epsilon_closure('q0', ...) -> {'q1', 'q0', 'q2'}  (expected {'q1', 'q0', 'q2'})

5/5 test cases passed for Question 1.


---
## 🚀 Main — Demonstration

This cell builds a small NFA (that uses epsilon-transitions) and runs the provided `nfa_accepts` engine on a few sample strings, so you can see your `epsilon_closure` function powering the full evaluation pipeline end-to-end.


In [7]:
# ===== MAIN DEMO =====
# NFA over alphabet {a, b} that accepts strings matching  a*b  (via epsilon-transitions)
demo_nfa = {
    'states': {'q0', 'q1', 'q2'},
    'alphabet': {'a', 'b'},
    'transitions': {
        ('q0', 'a'): {'q0'},
        ('q0', EPSILON): {'q1'},
        ('q1', 'b'): {'q2'},
    },
    'start': 'q0',
    'accept': {'q2'},
}

print("### Testing demo_nfa (accepts strings of the form a*b) ###")
demo_strings = ['b', 'ab', 'aab', 'aaab', 'a', 'ba', '']
for s in demo_strings:
    print(f"  nfa_accepts(demo_nfa, {s!r:7s}) -> {nfa_accepts(demo_nfa, s)}")


### Testing demo_nfa (accepts strings of the form a*b) ###
  nfa_accepts(demo_nfa, 'b'    ) -> True
  nfa_accepts(demo_nfa, 'ab'   ) -> True
  nfa_accepts(demo_nfa, 'aab'  ) -> True
  nfa_accepts(demo_nfa, 'aaab' ) -> True
  nfa_accepts(demo_nfa, 'a'    ) -> False
  nfa_accepts(demo_nfa, 'ba'   ) -> False
  nfa_accepts(demo_nfa, ''     ) -> False


Epsilon-closure represents all states that an NFA can reach from a given state without consuming an input symbol, including the state itself. Tracking sets of states through epsilon-closures is an essential part of the Subset Construction Algorithm because each set of NFA states can represent one state of the equivalent DFA.